In [ ]:
%pip install --force-reinstall --no-cache-dir "self_fourier_shell_correlation @ git+https://github.com/vicente-gonzalez-ruiz/self_fourier_shell_correlation"

In [ ]:
%pip show self_fourier_shell_correlation

In [ ]:
%pip install --force-reinstall --no-cache-dir "shuffling @ git+https://github.com/vicente-gonzalez-ruiz/shuffling"

In [ ]:
#%pip install opencv-python

In [ ]:
#%pip install "motion_estimation @ git+https://github.com/vicente-gonzalez-ruiz/motion_estimation"

In [ ]:
from pathlib import Path
from collections import namedtuple
import json
import numpy as np

In [ ]:
import os; cwd = os.getcwd(); print("Current Working Directory:", cwd)

In [ ]:
from pathlib import Path; cwd = Path.cwd(); print("Current Working Directory:", cwd)

In [ ]:
Args = namedtuple("args", ["original", "EO", "REO"])
args = Args("/home/jupyter-jjfdez/tomograms/empiar_11275.mrc", "even_odd/denoised_vol/empiar_11275.mrc", "even_odd_registered/denoised_vol/empiar_11275.mrc")

In [ ]:
def read_MRC(file_path):
    return mrcfile.read(file_path)

In [ ]:
import sys
print(sys.executable)

In [ ]:
from self_fourier_shell_correlation import fsc_utils as fsc

In [ ]:
import mrcfile

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
orig = read_MRC(args.original)

In [ ]:
#EO = read_MRC(args.EO)
import numpy as np
import mrcfile

file_path = args.EO

with mrcfile.open(file_path, permissive=True) as m:
    nx = m.header.nx
    ny = m.header.ny
    nz = m.header.nz
    nsymbt = m.header.nsymbt

# compute data offset
offset = 1024 + nsymbt

# read raw bytes manually
data = np.fromfile(
    file_path,
    dtype=np.float32,   # <-- force desired dtype
    offset=offset
)

# reshape manually
EO = data.reshape((nz, ny, nx))

In [ ]:
#REO = read_MRC(args.REO)
import numpy as np
import mrcfile

file_path = args.REO

with mrcfile.open(file_path, permissive=True) as m:
    nx = m.header.nx
    ny = m.header.ny
    nz = m.header.nz
    nsymbt = m.header.nsymbt

# compute data offset
offset = 1024 + nsymbt

# read raw bytes manually
data = np.fromfile(
    file_path,
    dtype=np.float32,   # <-- force desired dtype
    offset=offset
)

# reshape manually
REO = data.reshape((nz, ny, nx))

In [ ]:
list_fsc_values__orig = []
list_fsc_values__EO = []
list_fsc_values__REO = []
for i in range(orig.shape[0]):
    print(i, '/', orig.shape[0])
    spatial_freqs, fsc_values__orig = fsc.get_SFRC_curve__subsampled_chessboard(orig[i])
    spatial_freqs, fsc_values__EO = fsc.get_SFRC_curve__subsampled_chessboard(EO[i])
    spatial_freqs, fsc_values__REO = fsc.get_SFRC_curve__subsampled_chessboard(REO[i])
    list_fsc_values__orig.append(fsc_values__orig)
    list_fsc_values__EO.append(fsc_values__EO)
    list_fsc_values__REO.append(fsc_values__REO)

In [ ]:
avg_fsc_values__orig = np.mean(list_fsc_values__orig, axis=0)
avg_fsc_values__EO = np.mean(list_fsc_values__EO, axis=0)
avg_fsc_values__REO = np.mean(list_fsc_values__REO, axis=0)

In [ ]:
plt.title("empiar_11275")
plt.xlabel("Normalized Spatial Frequency (cycles/pixel)")
plt.ylabel("Average Self Fourier Ring Correlation")
plt.plot(spatial_freqs, avg_fsc_values__orig, label="Noisy", color="blue")
plt.plot(spatial_freqs, avg_fsc_values__EO, label="N2N-EO", color="red")
plt.plot(spatial_freqs, avg_fsc_values__REO, label="N2N-REO", color="green")
plt.legend(loc='lower left')

In [ ]:
!/home/jupyter-jjfdez/oneimageFRC/bin/oneimagefrc -v {args.original} /tmp/0
!/home/jupyter-jjfdez/oneimageFRC/bin/oneimagefrc -v {args.EO} /tmp/1
!/home/jupyter-jjfdez/oneimageFRC/bin/oneimagefrc -v {args.REO} /tmp/2

In [ ]:
def read_FRC(f):
    x = []
    y = []

    with open(f, 'r') as file:
        for line in file:
            parts = line.split()
            if len(parts) == 2:
                x.append(float(parts[0]))
                y.append(float(parts[1]))
    return x, y

In [ ]:
x_orig, y_orig = read_FRC("/tmp/0")
x_EO, y_EO = read_FRC("/tmp/1")
x_REO, y_REO = read_FRC("/tmp/2")

In [ ]:
plt.title("empiar_11275")
plt.xlabel("Normalized Spatial Frequency (cycles/pixel)")
plt.ylabel("Average Self Fourier Ring Correlation")
plt.plot(x_orig, y_orig, label="Noisy", color="blue")
plt.plot(x_EO, y_EO, label="N2N-EO", color="red")
plt.plot(x_REO, y_REO, label="N2N-REO", color="green")
plt.legend(loc='lower left')

In [ ]:
freq, orig, EO, REO = np.loadtxt("curvas_empiar_11275.asfrc", unpack=True)

In [ ]:
plt.title("empiar_11275")
plt.xlabel("Normalized Spatial Frequency")
plt.ylabel("Average Self Fourier Ring Correlation")
plt.plot(freq, orig, label="Noisy", color="blue")
plt.plot(freq, EO, label="N2N-EO", color="red")
plt.plot(freq, REO, label="N2N-REO", color="green")
plt.legend(loc='lower left')